### Import Dependencies

In [ ]:
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from openai import AsyncOpenAI

from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
# from ragas.embeddings import OpenAIEmbeddings

from ragas.dataset_schema import SingleTurnSample

from ragas.metrics.collections import Faithfulness, AnswerRelevancy

from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall

### Download an example reference data point from LangSmith

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
ls_client = Client()

In [ ]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [ ]:
dataset

In [ ]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].outputs

In [ ]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].inputs

In [ ]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[25].outputs

### RAG Pipeline

In [ ]:
qdrant_client = QdrantClient("url=http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, query):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(query, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [ ]:
rag_pipeline("Can I get a charger?", qdrant_client)

### RAGAS Metrics

In [ ]:
reference_input

In [ ]:
reference_output

In [ ]:
result = rag_pipeline(reference_input["question"], qdrant_client)

In [ ]:
result

In [ ]:
openai_client = AsyncOpenAI()

In [ ]:
ragas_llm = llm_factory("gpt-5.4-mini", client=openai_client, max_completion_tokens=4000)

args = ragas_llm.model_args
args.pop("max_tokens", None)
args.pop("top_p", None)
args["temperature"] = 0.0
args["max_completion_tokens"] = 4000

ragas_embeddings = embedding_factory("openai", model="text-embedding-3-small", client=openai_client)

In [ ]:
async def ragas_faitfulness(run, example):

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

In [ ]:
await ragas_faitfulness(result, reference_output)

In [ ]:
async def ragas_relevancy(run, example):

    scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"]
    )

In [ ]:
await ragas_relevancy(result, reference_output)

In [ ]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_precision_id_based(result, reference_output)

In [ ]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_output)